# Sparta Day cleaning
Reads the 152 Sparta Day .txt files from the S3 bucket download, cleans them,
and saves one table: sparta_day_clean.csv (4,134 rows).

In [17]:
# Unzip the bucket download from Google Drive
from google.colab import drive
drive.mount("/content/drive")
!unzip -q -o /content/drive/MyDrive/bucket_data.zip -d bucket_data
!ls bucket_data

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Academy  Talent


In [18]:
import glob
import re
import pandas as pd

# Each Sparta Day is one .txt file: line 1 = date, line 2 = location, then one line per person
sparta_texts = {p: open(p).read() for p in glob.glob("bucket_data/Talent/*.txt")}
print(len(sparta_texts), "Sparta Day files found")   # expecting 152

152 Sparta Day files found


In [19]:
# Example line: "TONIA GOODIE -  Psychometrics: 57/100, Presentation: 21/32"
LINE = re.compile(
    r"^(?P<name>.+?)\s*-\s*Psychometrics:\s*(?P<psychometric_score>\d+)\s*/\s*(?P<psychometric_max>\d+)"
    r"\s*,\s*Presentation:\s*(?P<presentation_score>\d+)\s*/\s*(?P<presentation_max>\d+)\s*$"
)

rows, unreadable = [], []
for path, text in sparta_texts.items():
    lines = [l.strip() for l in text.splitlines() if l.strip()]   # drop blank lines
    date_line, location_line, people = lines[0], lines[1], lines[2:]
    for line in people:
        match = LINE.match(line)
        if match:
            rows.append({"source_file": path, "date_raw": date_line,
                         "location_raw": location_line, **match.groupdict()})
        else:
            unreadable.append((path, line))

raw_sparta_df = pd.DataFrame(rows)   # raw copy, never edited
print(len(raw_sparta_df), "people read")          # expect 4134
print(len(unreadable), "lines not understood")    # expect 0
raw_sparta_df.head()

4134 people read
0 lines not understood


,source_file,date_raw,location_raw,name,psychometric_score,psychometric_max,presentation_score,presentation_max
0,bucket_data/Talent/Sparta Day 7 March 2019.txt,Thursday 7 March 2019,Birmingham Academy,CHRISTYNA THIRLWELL,56,100,25,32
1,bucket_data/Talent/Sparta Day 7 March 2019.txt,Thursday 7 March 2019,Birmingham Academy,RANDEE MACKISON,52,100,22,32
2,bucket_data/Talent/Sparta Day 7 March 2019.txt,Thursday 7 March 2019,Birmingham Academy,DOT MCGINLY,55,100,14,32
3,bucket_data/Talent/Sparta Day 7 March 2019.txt,Thursday 7 March 2019,Birmingham Academy,INGELBERT FRI,64,100,17,32
4,bucket_data/Talent/Sparta Day 7 March 2019.txt,Thursday 7 March 2019,Birmingham Academy,ZACHARIAS ARNAUDUC,51,100,17,32


In [20]:
df = raw_sparta_df.copy()

# 1. Score maximums: should only be 100 and 32
print("Psychometric max values:", df["psychometric_max"].unique())
print("Presentation max values: ", df["presentation_max"].unique())

# 2. Dates: do they all convert?
df["date"] = pd.to_datetime(df["date_raw"], format="%A %d %B %Y", errors="coerce")
print("\nDates that didn't convert:", df.loc[df["date"].isna(), "date_raw"].unique())
print("Date range:", df["date"].min(), "to", df["date"].max())

# 3. Does the date in the file name match line 1?
file_date = pd.to_datetime(df["source_file"].str.extract(r"(\d{1,2} \w+ \d{4})")[0],
                           format="%d %B %Y", errors="coerce")
print("Rows where filename date doesn't match:", (file_date != df["date"]).sum())

# 4. Locations
print("\nLocations:")
print(df["location_raw"].value_counts())

# 5. Scores above the maximum
print("\nScores above max:",
      ((df["psychometric_score"].astype(int) > df["psychometric_max"].astype(int)) |
       (df["presentation_score"].astype(int) > df["presentation_max"].astype(int))).sum())

# 6. Duplicates
print("Same name on the same day:", df.duplicated(subset=["name", "date_raw"]).sum())
print("Same name on different days:", df["name"].duplicated(keep=False).sum() -
      df.duplicated(subset=["name", "date_raw"], keep=False).sum())

Psychometric max values: ['100']
Presentation max values:  ['32']

Dates that didn't convert: []
Date range: 2019-01-02 00:00:00 to 2019-12-19 00:00:00
Rows where filename date doesn't match: 0

Locations:
location_raw
London Academy        2782
Birmingham Academy    1352
Name: count, dtype: int64

Scores above max: 0
Same name on the same day: 0
Same name on different days: 4


In [21]:
# Names: collapse extra spaces, trim, ALL CAPS -> Title Case
df["name"] = (df["name"].str.replace(r"\s+", " ", regex=True)
                        .str.strip()
                        .str.title())

# Two typos found in the raw files (checked against the applicant CSVs)
df["name"] = df["name"].replace({
    "Keen Bentham3": "Keen Bentham",          # stray digit
    "L;Urette Daveley": "L'Urette Daveley",   # semicolon instead of apostrophe
})

# Lowercase copy of the name, used to match people to other tables
df["name_key"] = df["name"].str.lower()

# "London Academy" -> "London"
df["location"] = df["location_raw"].str.replace("Academy", "", regex=False).str.strip()

# Scores: text -> numbers
for col in ["psychometric_score", "presentation_score"]:
    df[col] = df[col].astype(int)

sparta_clean = df[["name", "name_key", "date", "location",
                   "psychometric_score", "presentation_score", "source_file"]]
sparta_clean.head()

,name,name_key,date,location,psychometric_score,presentation_score,source_file
0,Christyna Thirlwell,christyna thirlwell,2019-03-07,Birmingham,56,25,bucket_data/Talent/Sparta Day 7 March 2019.txt
1,Randee Mackison,randee mackison,2019-03-07,Birmingham,52,22,bucket_data/Talent/Sparta Day 7 March 2019.txt
2,Dot Mcginly,dot mcginly,2019-03-07,Birmingham,55,14,bucket_data/Talent/Sparta Day 7 March 2019.txt
3,Ingelbert Fri,ingelbert fri,2019-03-07,Birmingham,64,17,bucket_data/Talent/Sparta Day 7 March 2019.txt
4,Zacharias Arnauduc,zacharias arnauduc,2019-03-07,Birmingham,51,17,bucket_data/Talent/Sparta Day 7 March 2019.txt


In [22]:
same_name = df[df["name"].duplicated(keep=False)].sort_values("name")
same_name[["name", "date", "location_raw", "psychometric_score", "presentation_score"]]

,name,date,location_raw,psychometric_score,presentation_score
1900,Nicolette Bonehill,2019-02-05,London Academy,52,23
3684,Nicolette Bonehill,2019-11-19,London Academy,47,20
1823,Shurlocke Cringle,2019-05-21,London Academy,52,21
3243,Shurlocke Cringle,2019-01-31,Birmingham Academy,49,18


In [23]:
sparta_clean.isna().sum()

,0
name,0
name_key,0
date,0
location,0
psychometric_score,0
presentation_score,0
source_file,0


In [24]:
   # Final checks - the notebook stops here with an error if anything is wrong
   assert len(sparta_clean) == 4134, "wrong row count"
   assert sparta_clean.isna().sum().sum() == 0, "blanks found"
   assert sparta_clean["name"].str.match(r"^[A-Za-z' .-]+$").all(), "odd characters in names"
   assert sparta_clean["psychometric_score"].between(0, 100).all(), "psychometric out of range"
   assert sparta_clean["presentation_score"].between(0, 32).all(), "presentation out of range"
   assert not sparta_clean.duplicated(subset=["name", "date"]).any(), "duplicates found"
   print("All checks passed")

All checks passed


In [25]:
   # Save the clean table to Google Drive
   sparta_clean.to_csv("/content/drive/MyDrive/sparta_day_clean.csv", index=False)
   print("Saved", len(sparta_clean), "rows")

Saved 4134 rows
